# Coding Agent

You'll need to build a Coding Agent powered by an LLM that can:
- Clone and explore GitHub repositories
- Read, analyze, and modify code files
- Execute tasks autonomously based on natural language instructions

#### Create OpenAI Client

In [ ]:
import tiktoken

TOKENIZER = tiktoken.encoding_for_model("gpt-4o-mini")
MAX_CONTEXT_TOKENS = 6000  # dejamos margen del límite del modelo

def count_tokens(messages: list) -> int:
    """Cuenta tokens aproximados del historial."""
    total = 0
    for m in messages:
        content = m.get("content") or ""
        total += len(TOKENIZER.encode(str(content)))
    return total

def summarize_history(messages: list) -> list:
    """
    Si el historial es muy largo, resume los mensajes del medio
    y conserva el system prompt + últimos 4 mensajes.
    """
    system = [m for m in messages if m["role"] == "system"]
    rest   = [m for m in messages if m["role"] != "system"]

    if len(rest) <= 4:
        return messages  # no hace falta resumir

    to_summarize = rest[:-4]
    recent       = rest[-4:]

    history_text = "\n".join(
        f"{m['role'].upper()}: {str(m.get('content', ''))[:300]}"
        for m in to_summarize
    )

    summary_response = client.chat.completions.create(
        model=MODEL,
        messages=[{
            "role": "user",
            "content": f"Resumí en 3-5 oraciones qué se hizo hasta ahora:\n\n{history_text}"
        }]
    )
    summary = summary_response.choices[0].message.content

    summary_msg = {
        "role": "system",
        "content": f"[RESUMEN DE CONTEXTO PREVIO]\n{summary}"
    }

    print(f"  [CONTEXTO] Historial resumido ({len(to_summarize)} mensajes → 1 resumen)")
    return system + [summary_msg] + recent


def detect_loop(state: dict, tool_name: str, args: dict) -> bool:
    """
    Detecta si el agente está repitiendo la misma acción sin avanzar.
    Retorna True si detecta loop.
    """
    # buscar en el progreso cuántas veces se llamó esta tool con estos mismos args
    key = f"{tool_name}:{json.dumps(args, sort_keys=True)}"
    repetitions = sum(1 for p in state["progress"] if key in p)

    if repetitions >= 2:
        print(f"  [LOOP DETECTADO] '{tool_name}' repetido {repetitions+1} veces sin avance.")
        return True
    return False


def inner_loop_with_guards(messages: list, state: dict, tools_schema: list,
                            supervision: bool, subagent_name: str) -> str:
    """
    Inner loop extendido con detección de loops y manejo de contexto.
    Reemplaza al inner_loop original para los subagentes.
    """
    iteracion = 0

    while True:
        iteracion += 1

        # manejo de contexto largo
        if count_tokens(messages) > MAX_CONTEXT_TOKENS:
            messages = summarize_history(messages)

        print(f"  [{subagent_name.upper()}] iteración {iteracion}")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )

        msg = response.choices[0].message

        if not msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content})
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in msg.tool_calls
            ]
        })

        for call in msg.tool_calls:
            tool_name = call.function.name
            tool_args = json.loads(call.function.arguments)

            # detección de loop
            if detect_loop(state, tool_name, tool_args):
                loop_msg = (
                    f"Detecté que estoy repitiendo '{tool_name}' sin avanzar. "
                    f"Voy a cambiar de estrategia o detenerme."
                )
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": loop_msg
                })
                log_progress(state, subagent_name, f"LOOP DETECTADO en {tool_name} — cambiando estrategia")
                continue

            log_progress(state, subagent_name, f"tool: {tool_name} {list(tool_args.values())[:1]}")
            result = execute_tool(tool_name, tool_args, supervision)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
            })

print("✓ Guards listos: detección de loops + manejo de contexto")

In [ ]:
def run_main_agent(repo_url: str = None, repo_path: str = None, supervision: bool = True):
    """
    Agente principal. Recibe un repo (URL o path local),
    coordina los subagentes y genera el reporte de arquitectura.
    """

    # ── 1. clonar repo si se dio URL ──
    if repo_url and not repo_path:
        repo_name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
        repo_path = str(WORKSPACE / repo_name)

        if Path(repo_path).exists():
            print(f"✓ Repo ya clonado en {repo_path} (memoria)")
        else:
            print(f"Clonando {repo_url}...")
            result = subprocess.run(
                ["git", "clone", "--depth", "1", repo_url, repo_path],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                print(f"[ERROR] Clone falló:\n{result.stderr}")
                return
            print(f"✓ Repo clonado en {repo_path}")

    if not repo_path or not Path(repo_path).exists():
        print("[ERROR] Necesitás proveer un repo_url o un repo_path válido.")
        return

    request = f"Analizar el repositorio React en {repo_path} y generar reporte de arquitectura."

    # ── 2. verificar memoria previa ──
    if repo_path in PROJECT_MEMORY.get("architecture", {}):
        print(f"[MEMORIA] Ya analicé este repo antes. Usando contexto previo.")

    # ── 3. inicializar task_state ──
    state = new_task_state(request, repo_path)
    log_progress(state, "main", f"Tarea iniciada: {request}")

    # ── 4. traza Langfuse ──
    trace = lf.trace(
        name="react-architecture-agent",
        input={"repo_path": repo_path, "request": request},
        metadata={"model": MODEL, "supervision": supervision}
    )

    try:
        # ── 5. ejecutar subagentes en orden ──
        span_explorer = trace.span(name="explorer")
        run_explorer(state, supervision)
        span_explorer.end(output={"result": str(state["subagent_results"]["explorer"])[:500]})

        span_researcher = trace.span(name="researcher")
        run_researcher(state, supervision)
        span_researcher.end(output={
            "result": str(state["subagent_results"]["researcher"])[:500],
            "rag_chunks": len(state["rag_chunks_used"])
        })

        span_implementer = trace.span(name="implementer")
        run_implementer(state, supervision)
        span_implementer.end(output={"files_modified": state["files_modified"]})

        span_tester = trace.span(name="tester")
        run_tester(state, supervision)
        span_tester.end(output={"result": str(state["subagent_results"]["tester"])[:500]})

        span_reviewer = trace.span(name="reviewer")
        run_reviewer(state, supervision)
        span_reviewer.end(output={"result": str(state["subagent_results"]["reviewer"])[:500]})

        # ── 6. actualizar memoria persistente ──
        update_memory(PROJECT_MEMORY, state)

        # ── 7. resumen final ──
        print("\n" + "="*60)
        print("RESUMEN DE EJECUCIÓN")
        print("="*60)
        print(f"Repo:            {repo_path}")
        print(f"Archivos:        {state['files_modified']}")
        print(f"RAG chunks:      {len(state['rag_chunks_used'])}")
        print(f"Fuentes:         {list(set(s['source'] for s in state['sources_consulted']))}")
        print(f"Veredicto:       {state['subagent_results']['reviewer'][:200]}")
        print("\nProgreso:")
        for p in state["progress"]:
            print(f"  {p}")

        trace.update(
            output={"files_modified": state["files_modified"], "status": "completado"},
            metadata={"rag_chunks_used": len(state["rag_chunks_used"])}
        )

    except Exception as e:
        log_progress(state, "main", f"ERROR: {e}")
        trace.update(output={"status": "error", "error": str(e)})
        raise

    finally:
        lf.flush()

    return state

print("✓ Agente principal listo")

## Implementación

In [ ]:
#implementación de las herramientas

def read_file(path: str, **kwargs) -> str: #--> path --> leo documento
  try:
    with open(path, 'r', encoding='utf-8') as f:
      return f.read()
  except FileNotFoundError:
    return f"Error: el archivo no fue encontrado en '{path}'"
  except Exception as e:
    return f"Error al intentar leer '{path}': {e}"


def write_file(path: str, content:str) -> str: # escribo contenido en un archivo | reemplazo si ya exite
  try:
    Path(path).parent.mkdir(parents= True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
      f.write(content)
      return f"Archivo escrito con éxito en '{path}'"
  except Exception as e:
    return f"Hubo un error al escribir en '{path}': {e}"

def run_command(command: str) -> str: # --> ejecuto comando en terminal --> retorno stdout Y stderr
  try:
    result = subprocess.run(
        command,
        shell=True,
        capture_output = True,
        text=True,
        timeout=45
    )
    output = ""
    if result.stdout:
      output += f"\n\nSTDOUT:\n{result.stdout}"
    if result.stderr:
      output += f"\n\nSTDERR:\n{result.stderr}"
    output += f"\nReturn code: {result.returncode}"
    return output if output.strip() else "No hay output"
  except subprocess.TimeoutExpired:
    return "Error: el comando excedió el timeout de 45 segundos"
  except Exception as e:
    return f"Se produjo un error al ejecutar: {e}"

def list_files(directory:str=".") -> str: # listo un directorio, mínimo para que pueda operar
  try:
    p = Path(directory)
    if not p.exists():
      return "El directorio '{directory} especificado no existe"
    items = sorted(p.iterdir())
    lines = [f"Contenido de '{directory}': "]
    for item in items:
      lines.append(f"{item.name}")
    return "\n".join(lines) if len(lines) > 1 else f"'{directory}' está vacío"
  except Exception as e:
    return f"Error listando '{directory}': {e}"



def web_search(query:str)->str: # herramienta websearch itnegrada de openai
  try:
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": query}],
        tools=[{"type": "web_search_preview"}]

    )
    return response.choices[0].message.content
  except Exception as e:
    return f"Error en web_search: {e}"

In [ ]:
# Definición de herramientas

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Lee el contenido completo de un archivo dado su path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path al archivo a leer"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Escribe (o sobreescribe) contenido en un archivo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path del archivo a escribir"},
                    "content": {"type": "string", "description": "Contenido a escribir en el archivo"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Ejecuta un comando de terminal y devuelve stdout y stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Comando de terminal a ejecutar"}
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Lista archivos y carpetas en un directorio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {"type": "string", "description": "Path del directorio a listar (default: '.')"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Busca información en la web y devuelve los resultados.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Consulta de búsqueda"}
                },
                "required": ["query"]
            }
        }
    }
]

TOOLS_MAP = { # mapeo nombre --> función python
    "read_file": read_file,
    "write_file": write_file,
    "run_command": run_command,
    "list_files": list_files,
    "web_search": web_search,
}

DESTRUCTIVE_TOOLS = {"write_file", "run_command"} # necesita supervisión porque modifican el sistema

# esta es la lista de herramientas que puede ejecutar mi agente. El llm no puede ejecutar código directametne pero si
# pedir ejecutar una función. SIn este esquema, el modelo no puede usar las funciones ya que no sabe cuales existen
# tengo que aclarar cuáles hay, su nombre y los parámetros que speran

### Guardrails

In [ ]:
import json

guardrails_config = {
    "allowed_directories": ["/content/workspace"],
    "blocked_paths": ["/etc", "/root"],
    "blocked_commands": ["rm -rf", "git push", "sudo", "chmod"]
}

with open("guardrails.json", "w") as f:
    json.dump(guardrails_config, f, indent=2)

print("guardrails.json creado")

In [ ]:
# Funciones para crear archivos restringidos -> el agente no debiera poder acceder o modificarlos

# Archivo con permiso para todos -> el agente no debiera poder hacer chmod
# Intentar chmod 400 test.txt
def create_file_with_full_access(file_name, content):
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(file_name, 0o777)
    print(f"Archivo {file_name} creado con éxito y permisos totales.")

create_file_with_full_access("test.txt", "You shouldn't be able to chmod this!")

# Crear un directorio restringido
# Intentar que acceda
def crear_directorio(nombre_carpeta):
    try:
        # Crea la carpeta.
        # parents=True crea carpetas intermedias si no existen.
        # exist_ok=True evita errores si ya existe.
        os.makedirs(nombre_carpeta, exist_ok=True)
        print(f"Directorio '{nombre_carpeta}' listo.")
    except Exception as e:
        print(f"Error al crear directorio: {e}")

crear_directorio("root")

In [ ]:
def load_guardrails(path="guardrails.json") -> dict:
    try:
        with open(path) as f:
            config = json.load(f)
        print(f"Guardrails cargados: {config}")
        return config
    except FileNotFoundError:
        print("Sin guardrails.json, sin restricciones.")
        return {}

GUARDRAILS = load_guardrails()

### Loops

In [ ]:
PROMPT = """Sos un agente de código cuyo trabajo es ayudar al usaurio con sus tareas de código.
Podes usar las herramietnas disponibles, estas son: read_file, write_file, run_command, list_files y web_search.
Respetá los siguietnes pasos al recibir una tarea:
1) Analizá los requisitos y pasos necesarios para resolver el problema
2) Hacé uso de las tools, de forma iterativa, para cumplir los objetivos
3) Verificá que el trabajo hecho sea correcto con, por ejemplo, tests
4) Reportá el resultado al usuario y explicá cómo lo resolviste

Mantené al usuario siempre al tanto de qué y por qué hacés lo que hacés."""


def execute_tool(name: str, args: dict, supervision: bool) -> str: # dict --> []
  args = {k.strip().rstrip('?'): v for k, v in args.items()}

  error = validate_tool_call(name, args) # valido guardrails
  if error:
      print(error)
      return error  # el LLM se entera y busca otra forma

  if supervision and name in DESTRUCTIVE_TOOLS:
    message = name
    if name == "run_command":
      message += " " + " ".join(args.values())
    print(f"\n[SUPERVISIÓN] El agente quiere ejecutar: {message}")
    choice = input("¿Permitir? (s/n): ").strip().lower() # strip elimina por defecto los espacios en blanco, tabulaciones y saltos de línea
    if choice != 's':
      return f"Cancelando {name}..."
  func = TOOLS_MAP[name]
  result = func(**args) #** desempaqueta el diccionario
  return result

def inner_loop(messages: list, supervision:bool)-> str: # es el loop interno. Llama al LLM y ejecuta tools hasta que respodnda sin tool_calls. Return: mensaje final del asistnet
  iteracion = 0
  while True:
    iteracion += 1
    print(f"Loop interno - iteración: {iteracion}")

    response = client.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = TOOLS_SCHEMA,
        tool_choice = "auto"
    )

    msg = response.choices[0].message # primera rta del modelo


    if not msg.tool_calls: # no hay tool calls --> agente terminó turno
        messages.append({"role": "assistant", "content": msg.content})
        return msg.content

    messages.append({
    "role": "assistant",
    "content": msg.content or "",  # convierte null a string vacío
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments
            }
        }
        for tc in msg.tool_calls
    ]
}) # sí hay tool calls --> ejecuta y devuelve rta

    for call in msg.tool_calls:
        tool_name = call.function.name
        tool_args = json.loads(call.function.arguments)

        print(f"\nAgente quiere utilizar: {tool_name} {" ".join(tool_args.values())}")
        result = execute_tool(tool_name, tool_args, supervision)

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
        })


def plan_mode_flow(user_message: str, messages: list) -> bool: # armo plan y espero confirmación del usuario
  print("\n[PLAN] Generando plan...")
  plan_messages = messages + [{
      "role": "user",
      "content": (
          f"Tarea: {user_message}\n\n"
          "Antes de hacer cualquier acción, describí detalladamente el plan de pasos "
          "que seguirías para completar esta tarea. No ejecutes ninguna tool todavía, "
          "solo listá los pasos."
      ) # agrego prompt al historial
  }]

  plan_response = client.chat.completions.create(
      model=MODEL,
      messages=plan_messages,
  )
  plan = plan_response.choices[0].message.content
  print(f"Plan propuesto:\n{plan}")

  choice = input("\n¿Aprobás este plan? (s/n/modificar): ").strip().lower()
  if choice == 'n':
      print("Tarea cancelada.")
      return False
  elif choice == 'modificar':
      modification = input("Describí los cambios al plan: ").strip() # prompt de modificación
      messages[-1]["content"] += f"\n\nModificación al plan: {modification}"
  return True


def run_agent(): # loop externo, chat interactua con agente. COmandos: plan, supervision, reset, exit
  messages = [{"role": "system", "content": PROMPT}]
  plan_mode = True # prendido opor defecto
  supervision = True

  print("Agente listo")
  print("="*50)
  print(f"Comandos:\n/plan (des/activa el paso a paso) | \n/supervision (des/activa control sobre operaciones críticas) | \n/reset (borra el historial) | \n/exit (abandonar chat) |")
  print(f"Estado inicial → Plan mode: {'ON' if plan_mode else 'OFF'} | Supervisión: {'ON' if supervision else 'OFF'}")

  while True: # loop ext espera input de user
      try:
          user_input = input("Prompt: ").strip()
      except (KeyboardInterrupt, EOFError):
          print("\nError. Saliendo...")
          break

      if not user_input:
          continue

      # Comandos especiales
      if user_input == "/exit":
          print("¡Hasta luego!")
          break
      elif user_input == "/reset":
          messages = [{"role": "system", "content": PROMPT}]
          print("Historial reseteado.")
          continue
      elif user_input == "/plan":
          plan_mode = not plan_mode
          print(f"Plan mode: {'ON' if plan_mode else 'OFF'}")
          continue
      elif user_input == "/supervision":
          supervision = not supervision
          print(f"Supervisión: {'ON' if supervision else 'OFF'}")
          continue

      messages.append({"role": "user", "content": user_input}) # msg de user al hisotiral

      # muestro plan --> pido aprobación
      if plan_mode:
          approved = plan_mode_flow(user_input, messages[:-1])
          if not approved:
              messages.pop()  # saco el mensaje del usuario si se canceló
              continue

      print("\nAgente: ", end="", flush=True) # loop inst ejecuta tools hasta rta final
      try:
          response = inner_loop(messages, supervision)
          print(f"\nAgente: {response}\n")
      except Exception as e:
          print(f"\nError en el agente: {e}\n")

## Ejecución

In [ ]:
REPO_URL = "https://github.com/facbook/create-react-app"
state = run_main_agent(repo_url=REPO_URL, supervision=False)